In [1]:
# -*- coding: utf-8 -*-
"""GAN-GRU 精选四维特征在沪深300指数增强策略上的回测。

特征来自 2020-01-01 至 2023-03-31 的独立候选因子聚类结果；
本回测从 2023-04-28 开始，不使用回测期标签进行特征选择。
"""

import copy
import sys
from importlib import import_module

import numpy as np
import pandas as pd

from factor_lib.function.bigquant_function.strategies.index_enhancement_backtest import (
    run_index_enhancement_backtest,
)


sys.dont_write_bytecode = True


# =============================================================================
# 1. 与上一轮完全保持一致的回测参数
# =============================================================================

START_DATE = "2023-04-28"
END_DATE = "2025-08-29"
BENCHMARK = "000300.SH"
REBALANCE_INTERVAL = 20
INITIAL_CASH = 10_000_000


# =============================================================================
# 2. 候选因子筛选后的四维特征
# =============================================================================
#
# 筛选区间：2020-01-01 至 2023-03-31，历史点时沪深300成分股。
# 选择原则：优先保留 |HAC t| >= 2 的候选；前三项来自不同的高有效性
# 聚类代表组。mfd_sellamt_nd 与 id2_std_nm 的暴露相似度约为 0.38，
# 属于中等相似，且其资金流含义和正向 RankIC 能提供额外信息。
#
# 不在这里预先根据 RankIC 正负号翻转特征。GAN-GRU 在训练期内自行学习
# 每个特征与未来收益的方向及非线性关系；人为翻转并不会增加可用信息。
FEATURE_SPEC = [
    {
        "factor_name": "hml_r_std_nm",
        "params": {
            "n_months": 5,
            "trading_days_per_month": 21,
        },
        "feature_name": "hml_r_std_5m",
    },
    {
        "factor_name": "id2_std_nm",
        "params": {
            "n_months": 3,
            "trading_days_per_month": 21,
        },
        "feature_name": "id2_std_3m",
    },
    {
        "factor_name": "exp_wgt_return_nm",
        "params": {
            "n_months": 6,
            "trading_days_per_month": 21,
        },
        "feature_name": "exp_wgt_return_6m",
    },
    {
        "factor_name": "mfd_sellamt_nd",
        "params": {
            "n_days": 1,
        },
        "feature_name": "mfd_sellamt_1d",
    },
]


# =============================================================================
# 3. 与上一轮完全保持一致的滚动训练参数
# =============================================================================

TRAINING_CONFIG = {
    "sequence_length": 40,
    "training_window_days": 200,
    "retrain_interval_days": 100,
    "label_horizon_days": REBALANCE_INTERVAL,
    "validation_ratio": 0.20,
    "purge_trading_days": REBALANCE_INTERVAL,
    "minimum_validation_dates": 20,
    "minimum_rankic_stocks": 30,
    "minimum_training_samples": 1000,
    "latent_dim": 10,
    "discriminator_hidden_size": 32,
    "lambda_reconstruction": 0.5,
    "gan_epochs": 4,
    "gru_max_epochs": 20,
    "early_stop_patience": 4,
    "rankic_min_delta": 0.0001,
    "batch_size": 1024,
    "cpu_threads": 4,
    "hidden_size": 64,
    "num_layers": 2,
    "dropout": 0.20,
    "gan_learning_rate": 0.0002,
    "gru_learning_rate": 0.001,
    "random_seed": 42,
}


# =============================================================================
# 4. 与上一轮完全保持一致的自定义指数倾斜函数
# =============================================================================

def bounded_exponential_tilt(
    transformed_scores,
    strength=0.35,
    minimum_multiplier=0.70,
    maximum_multiplier=1.50,
):
    """将截面标准化的模型分数转换为有上下限的非负倾斜乘数。"""
    if not isinstance(transformed_scores, pd.Series):
        raise TypeError("transformed_scores 必须为 pandas.Series。")

    strength = float(strength)
    minimum_multiplier = float(minimum_multiplier)
    maximum_multiplier = float(maximum_multiplier)

    if not np.isfinite(strength) or strength < 0:
        raise ValueError("strength 必须为有限非负数。")
    if (
        not np.isfinite(minimum_multiplier)
        or not np.isfinite(maximum_multiplier)
        or minimum_multiplier < 0
        or minimum_multiplier > maximum_multiplier
        or maximum_multiplier <= 0
    ):
        raise ValueError("倾斜乘数上下限设置无效。")

    multipliers = np.exp(strength * transformed_scores.astype(float))
    multipliers = multipliers.clip(
        lower=minimum_multiplier,
        upper=maximum_multiplier,
    )
    if multipliers.isna().any() or not np.isfinite(multipliers).all():
        raise ValueError("自定义倾斜函数产生了非有限值。")
    return multipliers


# =============================================================================
# 5. 建立本次回测独立的滚动模型状态，并执行回测
# =============================================================================

gan_gru_module = import_module(
    "factor_lib.Factor Repository.machine_learning_factors.gan_gru_score"
)

# 回测使用独立内存状态；不读取或写入任何先前回测的模型文件。
model_state_provider = gan_gru_module.build_model_state_provider(
    persistence_mode="memory",
)

FACTOR_PARAMS = {
    "feature_spec": copy.deepcopy(FEATURE_SPEC),
    "model_state_provider": model_state_provider,
    "training_config": copy.deepcopy(TRAINING_CONFIG),
}

print("[GAN-GRU精选特征指数增强] 基准指数：", BENCHMARK)
print("[GAN-GRU精选特征指数增强] 基础特征数：", len(FEATURE_SPEC))
print("[GAN-GRU精选特征指数增强] 特征：", [item["feature_name"] for item in FEATURE_SPEC])
print("[GAN-GRU精选特征指数增强] 调仓间隔：20 个交易日")
print("[GAN-GRU精选特征指数增强] 训练参数：", TRAINING_CONFIG)

backtest_result = run_index_enhancement_backtest(
    start_date=START_DATE,
    end_date=END_DATE,
    reference_portfolio={
        "type": "index",
        "index_code": BENCHMARK,
    },
    factor_name="gan_gru_score",
    factor_params=FACTOR_PARAMS,
    signal_direction=1,
    construction_method="benchmark_tilt",
    construction_params={
        "score_transform": "zscore",
        "score_clip": 3.0,
        "tilt_function": "custom",
        "custom_tilt_function": bounded_exponential_tilt,
        "custom_tilt_params": {
            "strength": 0.35,
            "minimum_multiplier": 0.70,
            "maximum_multiplier": 1.50,
        },
    },
    rebalance_rule={
        "type": "fixed_interval",
        "interval_trading_days": REBALANCE_INTERVAL,
    },
    portfolio_constraints={
        "target_stock_exposure": 1.00,
        "min_stock_exposure": 1.00,
        "max_stock_weight": None,
        "max_active_weight": None,
        "max_turnover": None,
        "industry_active_weight_limit": None,
        "style_active_exposure_limit": None,
        "max_tracking_error": None,
    },
    risk_model={
        "industry_scheme": "sw2021_l1",
        "style_fields": [],
    },
    feasibility_policy={
        "mode": "strict",
        "stop_at_first_feasible": True,
        "final_action": "hold_previous",
    },
    execution_config={
        "order_price_field_buy": "open",
        "order_price_field_sell": "open",
        "volume_limit": 0.025,
        "slippage_value": 0.001,
        "weight_tolerance": 0.0005,
        "rebalance_on_index_reconstitution": False,
    },
    trading_costs={
        "buy_cost": 0.0003,
        "sell_cost": 0.0003,
        "min_cost": 5.0,
        "tax_ratio": 0.0005,
    },
    initial_cash=INITIAL_CASH,
    performance_benchmark=BENCHMARK,
    show_progress=True,
    progress_every=1,
)

# BigTrader 回测图由策略函数自动显示。
# 需要审计时再手动取消下列代码的注释：
# from IPython.display import display
# display(backtest_result["data_diagnostics"])
# display(backtest_result["rebalance_audit"])
# display(backtest_result["feasibility_audit"])
# display(backtest_result["execution_audit"].head())
# display(backtest_result["trade_audit"].head())


[GAN-GRU精选特征指数增强] 基准指数： 000300.SH
[GAN-GRU精选特征指数增强] 基础特征数： 4
[GAN-GRU精选特征指数增强] 特征： ['hml_r_std_5m', 'id2_std_3m', 'exp_wgt_return_6m', 'mfd_sellamt_1d']
[GAN-GRU精选特征指数增强] 调仓间隔：20 个交易日
[GAN-GRU精选特征指数增强] 训练参数： {'sequence_length': 40, 'training_window_days': 200, 'retrain_interval_days': 100, 'label_horizon_days': 20, 'validation_ratio': 0.2, 'purge_trading_days': 20, 'minimum_validation_dates': 20, 'minimum_rankic_stocks': 30, 'minimum_training_samples': 1000, 'latent_dim': 10, 'discriminator_hidden_size': 32, 'lambda_reconstruction': 0.5, 'gan_epochs': 4, 'gru_max_epochs': 20, 'early_stop_patience': 4, 'rankic_min_delta': 0.0001, 'batch_size': 1024, 'cpu_threads': 4, 'hidden_size': 64, 'num_layers': 2, 'dropout': 0.2, 'gan_learning_rate': 0.0002, 'gru_learning_rate': 0.001, 'random_seed': 42}
[BigQuant loader] 4/4（100.00%），依赖数据加载完成，当前 mfd_sellamt_1d，213,465 行，耗时 9.1s                                                                                                                          

[2026-08-17 10:28:55] [info     ] bigtrader.v35 运行完成 [2094.745s].
[指数增强回测] [9/9] 回测与审计结果整理完成 | 1/1 (100.0%) | 订单6,352条，成交3,176条 | 已耗时 2111.0s                                                                                                                                                                     
